# Content-Based Movie Recommendation System

## 1. Environment Setup & Data Loading
Import required Python libraries for data processing and machine learning, and load datasets.


In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load datasets
movies = pd.read_csv("data/tmdb_5000_movies.csv")
credits = pd.read_csv("data/tmdb_5000_credits.csv")

print(f"Movies dataset shape: {movies.shape}")
print(f"Credits dataset shape: {credits.shape}")


### Merging Datasets & Filtering Columns
Merge `movies` and `credits` dataframes on `id` and select key feature columns relevant for content analysis and metadata.


In [ ]:
# Rename credit movie_id column to match movies id
credits = credits.rename(columns={"movie_id": "id"})

# Merge cast and crew into movies dataframe
movies = movies.merge(credits[["id", "cast", "crew"]], on="id", how="inner")

# Select relevant columns for content recommendation
movies = movies[
    ["id", "title", "genres", "keywords", "overview", "cast", "crew", "vote_average", "vote_count", "popularity"]
]

print("Merged dataset shape:", movies.shape)
display(movies.head(3))


## 2. Data Cleaning & Metadata Parsing
Handle missing values across text fields and convert JSON-stringified metadata into native Python lists.


In [ ]:
# Handle missing values in text columns
movies["overview"] = movies["overview"].fillna("")
movies["genres"] = movies["genres"].fillna("[]")
movies["keywords"] = movies["keywords"].fillna("[]")
movies["cast"] = movies["cast"].fillna("[]")
movies["crew"] = movies["crew"].fillna("[]")

def parse_features(text):
    try:
        return ast.literal_eval(text)
    except (ValueError, SyntaxError, TypeError):
        return []

for feature in ["genres", "keywords", "cast", "crew"]:
    movies[feature] = movies[feature].apply(parse_features)

print("Metadata columns parsed successfully.")


### Feature Extraction & Normalization
Extract key details:
- Director from crew array
- Top 3 cast members
- Top 10 genres and keywords

Names are normalized by removing spaces (e.g., `"Johnny Depp"` -> `"johnnydepp"`) so that full entity names are processed as single unique tokens by TF-IDF.


In [ ]:
def get_director(crew):
    """Extract director name from crew metadata."""
    for person in crew:
        if person.get("job") == "Director":
            return person.get("name", "")
    return ""

def get_top_names(items, limit=3):
    """Extract top N names from parsed dictionary list."""
    if not isinstance(items, list):
        return []
    return [item.get("name") for item in items[:limit] if item.get("name")]

def normalize_names(names):
    """Remove spaces and convert names to lowercase to preserve full entity names."""
    if isinstance(names, list):
        return [name.replace(" ", "").lower() for name in names]
    elif isinstance(names, str):
        return names.replace(" ", "").lower()
    return ""

# Extract metadata
movies["director"] = movies["crew"].apply(get_director)
movies["cast"] = movies["cast"].apply(lambda x: get_top_names(x, 3))
movies["genres"] = movies["genres"].apply(lambda x: get_top_names(x, 10))
movies["keywords"] = movies["keywords"].apply(lambda x: get_top_names(x, 10))

# Normalize names and tags
movies["genres"] = movies["genres"].apply(normalize_names)
movies["keywords"] = movies["keywords"].apply(normalize_names)
movies["cast"] = movies["cast"].apply(normalize_names)
movies["director"] = movies["director"].apply(normalize_names)

display(movies[["title", "cast", "director", "keywords", "genres"]].head(5))


## 3. Baseline Content-Based Recommendation Model
Combine cleaned features into a single text representation, convert them into TF-IDF numerical vectors, and compute pairwise cosine similarity scores.


In [ ]:
# Construct unified feature string for baseline model
movies["combined_features"] = (
    movies["genres"].apply(lambda x: " ".join(x)) + " " +
    movies["keywords"].apply(lambda x: " ".join(x)) + " " +
    movies["cast"].apply(lambda x: " ".join(x)) + " " +
    movies["director"] + " " +
    movies["overview"]
)

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words="english", max_features=10000)
tfidf_matrix = tfidf.fit_transform(movies["combined_features"])

# Compute pairwise cosine similarity matrix
similarity_matrix = cosine_similarity(tfidf_matrix)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Similarity matrix shape: {similarity_matrix.shape}")


### Baseline Recommendation Function
Create title-to-index lookup mapping and define recommendation function for unweighted model.


In [ ]:
# Series mapping lowercase movie titles to dataframe index
movie_indices = pd.Series(movies.index, index=movies["title"].str.lower()).drop_duplicates()

def recommend(movie_title, num_recommendations=10):
    """Retrieve top N similar movies using baseline TF-IDF similarity."""
    title_clean = movie_title.lower().strip()
    
    if title_clean not in movie_indices:
        print(f"Movie '{movie_title}' not found in dataset.")
        return
    
    idx = movie_indices[title_clean]
    similarity_scores = list(enumerate(similarity_matrix[idx]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    similarity_scores = similarity_scores[1:num_recommendations + 1]
    
    print(f"\nBaseline Recommendations for: {movies.iloc[idx]['title']}")
    print("-" * 55)
    for rank, (movie_idx, score) in enumerate(similarity_scores, start=1):
        title = movies.iloc[movie_idx]["title"]
        print(f"{rank:2d}. {title:<40} | Similarity: {score:.3f}")

# Test baseline recommendation
recommend("The Dark Knight")


## 4. Enhanced Feature-Weighted Recommendation Model
To improve recommendation quality, assign higher weights to key attributes (Genres: 3x, Director: 3x, Keywords: 2x, Cast: 2x, Overview: 1x).


In [ ]:
def create_weighted_features(row):
    """Combine text features with relative importance multipliers."""
    genres = " ".join(row["genres"])
    keywords = " ".join(row["keywords"])
    cast = " ".join(row["cast"])
    director = row["director"]
    overview = row["overview"]
    
    return (
        (genres + " ") * 3 +
        (keywords + " ") * 2 +
        (cast + " ") * 2 +
        (director + " ") * 3 +
        overview
    )

movies["weighted_features"] = movies.apply(create_weighted_features, axis=1)

# Fit weighted TF-IDF matrix
weighted_tfidf = TfidfVectorizer(stop_words="english", max_features=10000)
weighted_matrix = weighted_tfidf.fit_transform(movies["weighted_features"])

# Calculate weighted cosine similarity matrix
weighted_similarity = cosine_similarity(weighted_matrix)

print(f"Weighted TF-IDF matrix shape: {weighted_matrix.shape}")
print(f"Weighted similarity matrix shape: {weighted_similarity.shape}")


### Weighted Recommendation Function
Define recommendation engine based on weighted feature matrix and compare results.


In [ ]:
def recommend_weighted(movie_title, num_recommendations=10):
    """Retrieve top N similar movies using weighted feature similarity matrix."""
    title_clean = movie_title.lower().strip()
    
    if title_clean not in movie_indices:
        print(f"Movie '{movie_title}' not found in dataset.")
        return
    
    idx = movie_indices[title_clean]
    similarity_scores = list(enumerate(weighted_similarity[idx]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    similarity_scores = similarity_scores[1:num_recommendations + 1]
    
    print(f"\nWeighted Recommendations for: {movies.iloc[idx]['title']}")
    print("-" * 55)
    for rank, (movie_idx, score) in enumerate(similarity_scores, start=1):
        title = movies.iloc[movie_idx]["title"]
        print(f"{rank:2d}. {title:<40} | Similarity: {score:.3f}")

# Test weighted recommendation
recommend_weighted("The Dark Knight")


## 5. Dataset Overview & Interactive Query System
Provide dataset summary metrics and an interactive search function.


In [ ]:
print("MOVIE RECOMMENDATION SYSTEM - DATASET SUMMARY")
print("=" * 50)
print(f"Total Movies Processed : {len(movies):,}")
print(f"Total Unique Genres   : {len(set(g for genres in movies['genres'] for g in genres)):,}")
print(f"Average Rating        : {movies['vote_average'].mean():.2f} / 10")
print(f"Average Popularity    : {movies['popularity'].mean():.2f}")
print(f"TF-IDF Vocabulary Size: {len(weighted_tfidf.get_feature_names_out()):,}")
print(f"Similarity Matrix Size: {weighted_similarity.shape}")


### Interactive Recommendation Search
Functions for title searching (exact and partial match) and interactive prompt recommendation.


In [ ]:
def find_movie(title):
    """Find exact or partial movie match in dataset."""
    title = title.lower().strip()
    
    if title in movie_indices:
        return movie_indices[title]
    
    # Check partial title matches
    matches = [t for t in movie_indices.index if title in t]
    if matches:
        print(f"\nPartial matches found for '{title}':")
        for i, match in enumerate(matches[:5], start=1):
            print(f"{i}. {movies.iloc[movie_indices[match]]['title']}")
        return movie_indices[matches[0]]
    
    return None

def interactive_recommend(movie_name=None):
    if not movie_name:
        movie_name = input("Enter a movie name: ")
    
    idx = find_movie(movie_name)
    if idx is None:
        print(f"No matches found for '{movie_name}'.")
        return
    
    similarity_scores = list(enumerate(weighted_similarity[idx]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)[1:11]
    
    print(f"\nTop Recommendations for: '{movies.iloc[idx]['title']}'")
    print("-" * 55)
    for rank, (movie_idx, score) in enumerate(similarity_scores, start=1):
        print(f"{rank:2d}. {movies.iloc[movie_idx]['title']:<40} | Similarity: {score:.3f}")

# Example interactive test run
interactive_recommend()
